# Personalised RecSys training
As the project is aimed at helping me spend my time more efficiently, I'd concluded that being fancy would contradict with the project's values; meaning that there is no purpose in collecting data for and training a general-purpose RecSys. Moreover, the model itself wouldn't work so well for my specifical preferences. If you're playing to implement the same thing, make sure to construct your own dataset (you can find the original in the `news.csv` file).<br>
Bucke up. Let's import all the essential libraries first.

In [ ]:
import pandas as pd # For working with the .csv dataset
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel # Stuff that will help us vectorize text
from torch.utils.data import Dataset, DataLoader


Now, let's build the model itself. 

In [ ]:

class InterestRate(nn.Module):
    def __init__(self, bert_name, tokenizer_name=None):
        super(InterestRate, self).__init__()
        if not tokenizer_name:
            tokenizer_name = bert_name
        self.tokenizer = BertTokenizer.from_pretrained(tokenizer_name)
        self.bert = BertModel.from_pretrained(bert_name)
        self.fc1 = nn.Linear(768, 100)
        self.do = nn.Dropout(0.5)
        self.fc2 = nn.Linear(100, 10)
        self.fc3 = nn.Linear(10, 1)
        for param in self.bert.parameters(): # Very important. This loop tells PyTorch to refrain from calculating each of Bert's weights' influence on the loss during backpropagation.
            param.requires_grad = False
    def forward(self, text : str):
        encoded_inputs = self.tokenizer(text, padding=True, truncation=True, return_tensors='pt', max_length=256)
        x = self.bert(encoded_inputs['input_ids'], encoded_inputs['attention_mask']).last_hidden_state[:, 0, :]
        x = self.fc1(x)
        x = self.do(x)
        x = self.fc2(x)
        x = self.fc3(x)
        return x

IRModel = InterestRate('DeepPavlov/rubert-base-cased') # RuBERT - BERT analog for Russian language
learning_rate = 1e-4
loss_fn = nn.L1Loss() # Tried MSE first, but with it the model got comfortable only outputing something in the middle of the values interval (1-5) for all types of input data.
optimizer = torch.optim.Adam(IRModel.parameters(), lr=learning_rate)
num_epochs = 10


Finally, let's initialize the dataset class and train the model.

In [ ]:
class InterestDataset(Dataset):
    def __init__(self, csv_file, num_rows, reverse=False):
        df = pd.read_csv(csv_file)
        if reverse:
            df = df.iloc[::-1]
        self.x = []
        self.y = []
        count = 0
        for i,r in df.iterrows():
            if not pd.isnull(r['interest_rate']):
                self.x.append(r['headline'].strip())
                self.y.append(torch.tensor([[r['interest_rate']]]))
                count+=1
                if count == num_rows:
                    break

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

ds_train = InterestDataset('news.csv', 384)
ds_val = InterestDataset('news.csv', 64, True)

dl_train = DataLoader(ds_train, batch_size=64, shuffle=True)
ds_val = DataLoader(ds_val)


for epoch in range(1, num_epochs+1):

    train_loss = 0
    count = 0
    for i, data in enumerate(ds_train):
        x,y = data
        output = IRModel(x)

        optimizer.zero_grad()

        loss = loss_fn(output, y)
        loss.backward()

        train_loss += loss.item()
        count+=1

        optimizer.step()
    

    print(f"\n\nTraining loss after epoch №{epoch}: {train_loss/count}")

    val_loss = 0
    count = 0
    for x,y in ds_val:
        output = IRModel(x)
        val_loss += loss_fn(output, y)
        count += 1
    print(f"Validation loss after epoch №{epoch}: {val_loss/count}\n\n")


In [5]:
torch.save(IRModel, 'IRModel.pt')